In [84]:
import numpy as np
import pandas as pd
import json
import copy
from pathlib import Path
from datetime import datetime

In [5]:
paths = {
    'mapping': {
        "baseline": {
            'supervised': '/Users/sven/workspace/emotion-embeddings/emocoder/target/mapping/baseline/test/overview.csv',
        } ,
        'proposed': {
            'supervised':  '/Users/sven/workspace/emotion-embeddings/emocoder/target/mapping/multitask/test/overview.csv',
        } 
    },
    'word': {
        'baseline': {
            'supervised': '/Users/sven/workspace/emotion-embeddings/emocoder/target/word/baseline/test/overview.csv',
            'zero-shot': '/Users/sven/workspace/emotion-embeddings/emocoder/target/word/zeroshotbaseline/test/overview.csv',
        },
        'proposed': {
            'supervised': '/Users/sven/workspace/emotion-embeddings/emocoder/target/word/augmented/test/overview.csv',
            'zero-shot': '/Users/sven/workspace/emotion-embeddings/emocoder/target/word/augzero/test/overview.csv',
        }
    },
    'text': {
        'baseline': {
            'supervised': '/Users/sven/workspace/emotion-embeddings/emocoder/target/text/baseline/test/overview.csv',
            'zero-shot': '/Users/sven/workspace/emotion-embeddings/emocoder/target/text/zeroshotbaseline/test/overview.csv',
        },
        'proposed': {
            'supervised': '/Users/sven/workspace/emotion-embeddings/emocoder/target/text/augmented/test/overview.csv',
            'zero-shot': '/Users/sven/workspace/emotion-embeddings/emocoder/target/text/augzero/test/overview.csv',
        }
    },
    'image': {
        'baseline': {
            'supervised': '/Users/sven/workspace/emotion-embeddings/emocoder/target/image/baseline/test/overview.csv',
            'zero-shot': '/Users/sven/workspace/emotion-embeddings/emocoder/target/image/zeroshotbaseline/test/overview.csv',
        },
        'proposed': {
            'supervised': '/Users/sven/workspace/emotion-embeddings/emocoder/target/image/augmented/test/overview.csv',
            'zero-shot': '/Users/sven/workspace/emotion-embeddings/emocoder/target/image/augzero/test/overview.csv',
        }
    },
}

In [70]:
datasets = {
    'word': ['en1', 'en2', 'es1', 'es2', 'de1', 'de2', 'pl1', 'pl2', 'tr1', 'tr2'],
    'text': ['EmoB', 'AffT'],
    'image': ['ferBE', 'ferVAD', 'afnBE', 'afnVA']
}

modalities = ['word', 'text', 'image']
                # TODO There seems to be a problme that the p/r/f values are not recorded 
                # for the mapping datasets in the overview table.
                # I will skip this for now and focus on the other modalities.


approaches = ['baseline', 'proposed']

scenarios = ['supervised', 'zero-shot']

In [95]:
dfs = copy.deepcopy(paths)
for modality in  dfs.keys():
    for approach in dfs[modality].keys():
        for scenario in dfs[modality][approach].keys():
            print(modality, approach, scenario)
            df = pd.read_csv(dfs[modality][approach][scenario], index_col=0)
            

            # reformat data frames:
            if modality == 'mapping':
                
                if approach == 'baseline' :
                    rename={
                        "anew_stevenson_baseline_vad_to_be5": "anew-stevenson",
                        "anew_stevenson_baseline_va_to_be5": "anewVA-stevenson",
                        "anew_stevenson_baseline_be5_to_vad": "stevenson-anew",
                        "anew_stevenson_baseline_be5_to_va": "stevenson-anewVA",
                        "FER-VAD-BE": "ferVAD-ferBE",
                        "FER-BE-VAD": "ferBE-ferVAD",
                        "AffectNet-VA-BE": "affectnetVA-affectnetBE",
                        "AffectNet-BE-VA": "affectnetBE-affectnetVA"
                    }
                    
                    rows = [row for row in df.index if row in rename.keys()]
                    df = df.loc[rows]
                    df.rename(rename, inplace=True)

                if approach == 'proposed':
                    rename = {"ANEW_Stevenson_vad_be5": "anew-stevenson",
                            "ANEW_Stevenson_be5_vad": "stevenson-anew",
                            "ANEW_VA_Stevenson_va_be5": "anewVA-stevenson",
                            "ANEW_VA_Stevenson_be5_va": "stevenson-anewVA",
                            "FER_BE_VAD_vad_be_fer13": "ferVAD-ferBE",
                            "FER_BE_VAD_be_fer13_vad": "ferBE-ferVAD",
                            "AffectNet_Mapping_va_be_affectnet": "affectnetVA-affectnetBE",
                            "AffectNet_Mapping_be_affectnet_va": "affectnetBE-affectnetVA"}
                    rows = [row for row in df.index if row in rename.keys()]
                    df = df.loc[rows]
                    df.rename(rename, inplace=True)

                # final row rename
                df.rename({
                    "anew-stevenson": 'en1-en2',
                    "anewVA-stevenson": 'en1*-en2',
                    "stevenson-anew": 'en2-en1',
                    "stevenson-anewVA": 'en2-en1*',
                    "ferVAD-ferBE": "ferVAD-ferBE",
                    'ferBE-ferVAD': 'ferBE-ferVAD',
                    "affectnetVA-affectnetBE": 'afnVA-afnBE',
                    'affectnetBE-affectnetVA': 'afnBE-afnVA'
                }, inplace=True)
            
            if modality == 'word':
                rename = {
                    'anew': 'en1',
                    'stevenson':'en2',
                    'stadthagenVA': 'es1',
                    'stadthagenBE': 'es2',
                    'vo': 'de1',
                    'briesemeister': 'de2',
                    'riegel': 'pl1',
                    'wierzba': 'pl2',
                    'kapucuVA': 'tr1',
                    'kapucuBE': 'tr2'
                }
                df.rename(rename, inplace=True)
                rows = datasets[modality]
                df = df.loc[rows]

            if modality == 'text':
                if approach == 'proposed' and scenario == 'supervised':
                    rename = {"EmoBank-augmented": "emobank", 
                                "CVAT-augmented": "cvat",
                                "AffText-augmented": "afftext"}
                    df.rename(rename, inplace=True)


                if approach == 'proposed' and scenario == 'zero-shot':
                    rename ={"emobank-augzero": "emobank", 
                    "afftext-augzero": "afftext"}
                    df.rename(rename, inplace=True)


                # final rename
                rename = {
                    'emobank': 'EmoB',
                    'afftext': 'AffT'
                }
                df.rename(rename, inplace=True)
                df = df.loc[datasets[modality]]


            if modality == 'image':
                
                if approach == 'baseline' and scenario == 'supervised':
                    rename = {"you2017_baseline": "you2017", "iaps_baseline": "iaps", 
                                     "fer2013vad_baseline": "fer_vad", "fer2013_be_baseline": "fer_be", 
                                     "affectnet_va_baseline": "affectnet_va", 
                                     "affectnet_be_baseline": "affectnet_be"}
                    df.rename(rename, inplace=True)

                if approach == 'baseline' and scenario == 'zero-shot':
                    df.rename(inplace=True, index={"fer_vad_be": "fer_be",
                                             "fer_be_vad": "fer_vad", 
                                             "affectnet_va_be": "affectnet_be", 
                                             "affectnet_be_va": "affectnet_va"})

                if approach == 'proposed' and scenario == 'zero-shot':
                    df.rename(inplace=True, index={"fer_va_be": "fer_be",
                                    "fer_be_va": "fer_vad", 
                                    "affectnet_va_be": "affectnet_be", 
                                    "affectnet_be_va": "affectnet_va"})

                # final rename
                df.rename({
                    'fer_vad': 'ferVAD',
                    'fer_be': 'ferBE',
                    'affectnet_va': 'afnVA',
                    'affectnet_be': 'afnBE',
                }, inplace=True)
                df = df.loc[datasets[modality]]



            print(df)
            print()
            print()
            dfs[modality][approach][scenario] = df

mapping baseline supervised
                   acc   valence   arousal  dominance       joy     anger  \
en1-en2            NaN       NaN       NaN        NaN  0.962410  0.872518   
en1*-en2           NaN       NaN       NaN        NaN  0.961838  0.870564   
en2-en1            NaN  0.971296  0.784504   0.863931       NaN       NaN   
en2-en1*           NaN  0.970941  0.784643        NaN       NaN       NaN   
ferVAD-ferBE  0.660884       NaN       NaN        NaN       NaN       NaN   
ferBE-ferVAD       NaN  0.868214  0.745084   0.759575       NaN       NaN   
afnVA-afnBE   0.820565       NaN       NaN        NaN       NaN       NaN   
afnBE-afnVA        NaN  0.933670  0.924264        NaN       NaN       NaN   

               sadness      fear   disgust  f1_disgust  ...  rec_surprise  \
en1-en2       0.903338  0.888306  0.803039         NaN  ...           NaN   
en1*-en2      0.900518  0.880885  0.800957         NaN  ...           NaN   
en2-en1            NaN       NaN       NaN     

In [96]:
metric_str_regression = " in Pearson's \\textit{r}"
classification_datasets = ['ferBE', 'afnBE']
now = datetime.now()
datetime_string = now.strftime("%Y-%m-%d %H:%M:%S")

print(f"% Generated LateX code from {datetime_string}")
print()
print()
for modality in modalities:
    for dataset in datasets[modality]:
        df = pd.DataFrame(columns=pd.MultiIndex.from_product([approaches, scenarios]))
        for approach in approaches:
            for scenario in scenarios:
                result = dfs[modality][approach][scenario].loc[dataset]
                df[(approach, scenario)] = result
        df = df[~df.isna().any(axis=1)]
        # if 'mean' in df.index:
        #     df.drop('mean', inplace=True)
        print(df.style.format("{:.3f}").to_latex(hrules=True, environment='table', caption = "Detailed results for dataset " + dataset + (metric_str_regression if dataset not in classification_datasets else "") + "."))
        print()
        print()
    

% Generated LateX code from 2025-05-09 11:00:38


\begin{table}
\caption{Detailed results for dataset en1 in Pearson's \textit{r}.}
\begin{tabular}{lrrrr}
\toprule
 & \multicolumn{2}{r}{baseline} & \multicolumn{2}{r}{proposed} \\
 & supervised & zero-shot & supervised & zero-shot \\
\midrule
valence & 0.917 & 0.926 & 0.931 & 0.928 \\
arousal & 0.698 & 0.653 & 0.720 & 0.701 \\
dominance & 0.834 & 0.790 & 0.836 & 0.803 \\
mean & 0.816 & 0.790 & 0.829 & 0.810 \\
\bottomrule
\end{tabular}
\end{table}



\begin{table}
\caption{Detailed results for dataset en2 in Pearson's \textit{r}.}
\begin{tabular}{lrrrr}
\toprule
 & \multicolumn{2}{r}{baseline} & \multicolumn{2}{r}{proposed} \\
 & supervised & zero-shot & supervised & zero-shot \\
\midrule
joy & 0.924 & 0.921 & 0.927 & 0.926 \\
anger & 0.898 & 0.831 & 0.895 & 0.831 \\
sadness & 0.901 & 0.823 & 0.892 & 0.840 \\
fear & 0.893 & 0.859 & 0.896 & 0.855 \\
disgust & 0.884 & 0.745 & 0.870 & 0.769 \\
mean & 0.900 & 0.836 & 0.896 & 0.844 \\
\botto